# Phase 1: Marengo Video Embeddings

Generate 512-dimensional multimodal embeddings for video samples using the **Twelve Labs Marengo 3.0** model.

**Pipeline:** Load dataset from HuggingFace &rarr; Upload each video to Twelve Labs &rarr; Generate fused asset-level embedding &rarr; Store in FiftyOne

**Key specs:**
- Model: `marengo3.0` (V2 Embed API)
- Output: 512-d float vector per video (fused visual + audio)
- Dataset: [Voxel51/Safe_and_Unsafe_Behaviours](https://huggingface.co/datasets/Voxel51/Safe_and_Unsafe_Behaviours)

## 1. Setup & Imports

In [2]:
import os
import time

import numpy as np
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub
from twelvelabs import TwelveLabs, VideoInputRequest, MediaSource

# Validate API key
assert os.environ.get("TWELVELABS_API_KEY"), (
    "Set TWELVELABS_API_KEY env var before running this notebook"
)

client = TwelveLabs(api_key=os.environ["TWELVELABS_API_KEY"])
print("Twelve Labs client initialized")

Twelve Labs client initialized


## 2. Load Dataset

Pull 10 video samples from the Safe & Unsafe Behaviours dataset on HuggingFace.

In [3]:
dataset = load_from_hub("Voxel51/Safe_and_Unsafe_Behaviours", max_samples=10)
print(f"Loaded {len(dataset)} samples")
print(f"Media type: {dataset.media_type}")
print(f"Sample fields: {dataset.get_field_schema().keys()}")
print(dataset)

Loading dataset
Importing samples...
 100% |███████████████████| 10/10 [5.9ms elapsed, 0s remaining, 1.7K samples/s]      
Importing frames...
 100% |█████████████████████| 0/0 [2.1ms elapsed, ? remaining, ? samples/s]  
Migrating dataset 'Voxel51/Safe_and_Unsafe_Behaviours' to v1.14.0
Loaded 10 samples
Media type: video
Sample fields: odict_keys(['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'ground_truth'])
Name:        Voxel51/Safe_and_Unsafe_Behaviours
Media type:  video
Num samples: 10
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.VideoMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_t

## 3. Inspect a Sample

Look at one video sample to understand the data structure before embedding.

In [4]:
sample = dataset.first()
print(f"Filepath: {sample.filepath}")
print(f"File size: {os.path.getsize(sample.filepath) / 1024:.1f} KB")
print(f"Existing fields: {list(sample.field_names)}")
print(sample)

Filepath: /Users/rishimule/fiftyone/huggingface/hub/Voxel51/Safe_and_Unsafe_Behaviours/data/0_tr1.mp4
File size: 25259.7 KB
Existing fields: ['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'ground_truth']
<Sample: {
    'id': '6979231a91651488c3f85b00',
    'media_type': 'video',
    'filepath': '/Users/rishimule/fiftyone/huggingface/hub/Voxel51/Safe_and_Unsafe_Behaviours/data/0_tr1.mp4',
    'tags': ['train'],
    'metadata': <VideoMetadata: {
        'size_bytes': 25865947,
        'mime_type': 'video/mp4',
        'frame_width': 1920,
        'frame_height': 1080,
        'frame_rate': 24.83,
        'total_frame_count': 348,
        'duration': 14.015304,
        'encoding_str': 'avc1',
    }>,
    'created_at': datetime.datetime(2026, 4, 3, 16, 32, 27, 878000),
    'last_modified_at': datetime.datetime(2026, 4, 3, 16, 32, 27, 878000),
    'ground_truth': <Classification: {
        'id': '6979231a91651488c3f85aff',
        'tags': [],
        'label': 'Safe

## 4. Generate Marengo Embeddings

For each video:
1. **Upload** the local file to Twelve Labs via `client.assets.create()`
2. **Embed** using the V2 API with fused visual+audio at asset scope
3. **Store** the 512-d vector as `sample["embedding"]`

In [5]:
success_count = 0
fail_count = 0
timings = []

for i, sample in enumerate(dataset, start=1):
    filepath = sample.filepath
    filename = os.path.basename(filepath)
    print(f"[{i}/{len(dataset)}] {filename} ... ", end="", flush=True)

    start = time.time()
    try:
        # Upload video
        with open(filepath, "rb") as f:
            asset = client.assets.create(method="direct", file=f)

        # Generate fused asset-level embedding
        response = client.embed.v_2.create(
            input_type="video",
            model_name="marengo3.0",
            video=VideoInputRequest(
                media_source=MediaSource(asset_id=asset.id),
                embedding_option=["visual", "audio"],
                embedding_scope=["asset"],
                embedding_type=["fused_embedding"],
            ),
        )

        embedding = response.data[0].embedding
        sample["embedding"] = embedding
        sample.save()

        elapsed = time.time() - start
        timings.append(elapsed)
        success_count += 1
        print(f"{len(embedding)}-d  ({elapsed:.1f}s)")

    except Exception as e:
        elapsed = time.time() - start
        timings.append(elapsed)
        fail_count += 1
        print(f"FAILED ({elapsed:.1f}s): {e}")

dataset.save()
print(f"\nDone: {success_count} succeeded, {fail_count} failed")

[1/10] 0_tr1.mp4 ... 512-d  (25.5s)
[2/10] 0_tr10.mp4 ... 512-d  (15.5s)
[3/10] 0_tr100.mp4 ... 512-d  (7.8s)
[4/10] 0_tr101.mp4 ... 512-d  (6.0s)
[5/10] 0_tr102.mp4 ... 512-d  (8.0s)
[6/10] 0_tr103.mp4 ... 512-d  (7.2s)
[7/10] 0_tr104.mp4 ... 512-d  (7.2s)
[8/10] 0_tr105.mp4 ... 512-d  (7.9s)
[9/10] 0_tr106.mp4 ... 512-d  (12.3s)
[10/10] 0_tr107.mp4 ... 512-d  (10.0s)

Done: 10 succeeded, 0 failed


## 5. Verify Embeddings

Confirm the stored embeddings have the correct shape and inspect their statistics.

In [6]:
# Collect all embeddings into a matrix
embeddings = []
for sample in dataset:
    if sample["embedding"] is not None:
        embeddings.append(sample["embedding"])

embeddings = np.array(embeddings)
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Value range: [{embeddings.min():.4f}, {embeddings.max():.4f}]")
print(f"Mean norm: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"Samples with embeddings: {len(embeddings)} / {len(dataset)}")

Embedding matrix shape: (10, 512)
Dtype: float64
Value range: [-0.1123, 0.1221]
Mean norm: 0.9998
Samples with embeddings: 10 / 10


## 6. Pairwise Similarity

Compute cosine similarity between all video pairs to see how the embeddings separate content.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings)

# Print as a labeled table
filenames = [
    os.path.basename(s.filepath)[:12]
    for s in dataset
    if s["embedding"] is not None
]

print("Cosine similarity matrix:\n")
header = "            " + "  ".join(f"{n:>12}" for n in filenames)
print(header)
for i, name in enumerate(filenames):
    row = "  ".join(f"{sim_matrix[i][j]:12.3f}" for j in range(len(filenames)))
    print(f"{name:>12}  {row}")

# Summary stats (excluding self-similarity diagonal)
mask = ~np.eye(len(filenames), dtype=bool)
off_diag = sim_matrix[mask]
print(f"\nOff-diagonal similarity — min: {off_diag.min():.3f}, "
      f"mean: {off_diag.mean():.3f}, max: {off_diag.max():.3f}")

Cosine similarity matrix:

               0_tr1.mp4    0_tr10.mp4   0_tr100.mp4   0_tr101.mp4   0_tr102.mp4   0_tr103.mp4   0_tr104.mp4   0_tr105.mp4   0_tr106.mp4   0_tr107.mp4
   0_tr1.mp4         1.000         0.988         0.995         0.994         0.992         0.996         0.992         0.994         0.992         0.988
  0_tr10.mp4         0.988         1.000         0.989         0.990         0.990         0.990         0.992         0.981         0.984         0.980
 0_tr100.mp4         0.995         0.989         1.000         0.998         0.997         0.998         0.995         0.995         0.990         0.984
 0_tr101.mp4         0.994         0.990         0.998         1.000         0.998         0.998         0.996         0.994         0.988         0.983
 0_tr102.mp4         0.992         0.990         0.997         0.998         1.000         0.997         0.996         0.994         0.987         0.982
 0_tr103.mp4         0.996         0.990         0.998   

## 7. Timing Summary

How long did each embedding take? This helps estimate cost for larger datasets.

In [8]:
timings_arr = np.array(timings)
print(f"Total time:   {timings_arr.sum():.1f}s")
print(f"Mean / video: {timings_arr.mean():.1f}s")
print(f"Min:          {timings_arr.min():.1f}s")
print(f"Max:          {timings_arr.max():.1f}s")
print(f"\nEstimate for 100 videos: ~{timings_arr.mean() * 100 / 60:.0f} min")
print(f"Estimate for 600 videos: ~{timings_arr.mean() * 600 / 60:.0f} min")

Total time:   107.4s
Mean / video: 10.7s
Min:          6.0s
Max:          25.5s

Estimate for 100 videos: ~18 min
Estimate for 600 videos: ~107 min


## 8. Launch FiftyOne App

Visualize the dataset in the FiftyOne App to browse videos and confirm the embedding field is populated.

In [10]:
session = fo.launch_app(dataset)


Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.14.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|

